<!-- Cell 1 -->
# AquaSelect -- Notebook 4: Conformal Prediction + AquaSelect-Conformal Fusion

**Purpose**: Add conformal prediction with formal coverage guarantees to the selective prediction framework.  
**Inputs**: Saved logits from Notebooks 1a/1b, results from Notebook 2 (frozen).  
**Key addition**: Statistical guarantee that the true class is in the prediction set with probability ≥ 1-α.  
**New method**: AquaSelect-Conformal fusion combines learned selection with conformal set size for improved selective prediction.

In [ ]:
# Cell 2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from collections import defaultdict
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Notebook 4: Conformal Prediction | Device: {device}")

In [ ]:
# Cell 3
# Configuration and load saved outputs 

CFG = {
    "num_classes": 20,
    "seeds": [9, 42, 2003],
    "alpha_levels": [0.01, 0.05, 0.10, 0.15, 0.20],
    "output_dir": "/kaggle/working",
    # RAPS parameters
    "raps_lambda": 0.05,   # penalty for including low-ranked classes
    "raps_k_reg": 4,       # rank threshold where penalty kicks in
}

PATHS = {
    "convnext_logits": "/kaggle/input/datasets/abcd/aqua20-convnext-models/convnext_val_test_outputs.pth",
    "deit_logits": "/kaggle/input/datasets/abcd/aqua20-deit-models/deit_val_test_outputs.pth",
    "nb2_results": "/kaggle/input/datasets/abcd/aqua-20-notebook-2-frozen/notebook2_all_results (1).pth",
}

# Load everything
convnext_data = torch.load(PATHS["convnext_logits"], map_location="cpu", weights_only=False)
deit_data = torch.load(PATHS["deit_logits"], map_location="cpu", weights_only=False)
nb2_results = torch.load(PATHS["nb2_results"], map_location="cpu", weights_only=False)

label_names = nb2_results["label_names"]

saved_logits = {
    "convnext_tiny": convnext_data,
    "deit_small": deit_data,
}

print("All data loaded.")
print(f"ConvNeXt val logits shape (seed 42): {convnext_data['val_logits'][42].shape}")
print(f"ConvNeXt test logits shape (seed 42): {convnext_data['test_logits'][42].shape}")
print(f"DeiT val logits shape (seed 42): {deit_data['val_logits'][42].shape}")
print(f"DeiT test logits shape (seed 42): {deit_data['test_logits'][42].shape}")
print(f"Label names: {label_names}")

# Extract learned temperatures from Notebook 2 results
temperatures = {}
for backbone_name in ["convnext_tiny", "deit_small"]:
    temperatures[backbone_name] = {}
    for seed in CFG["seeds"]:
        T = nb2_results["aquaselect_results"][backbone_name][seed]["temperature"]
        temperatures[backbone_name][seed] = T
    print(f"\n{backbone_name} temperatures:")
    for seed in CFG["seeds"]:
        print(f"  Seed {seed}: T = {temperatures[backbone_name][seed]:.4f}")

print(f"\nRAPS params: λ={CFG['raps_lambda']}, k_reg={CFG['raps_k_reg']}")

In [ ]:
# Cell 4
# RAPS Conformal Prediction (with temperature scaling support) 

def compute_raps_scores(probs, labels, lam=0.05, k_reg=4):
    """Compute RAPS nonconformity scores.
    
    Score = cumulative_prob_at_true_class + λ * max(0, rank_of_true_class - k_reg)
    
    Args:
        probs: (N, C) softmax probabilities (ideally temperature-scaled)
        labels: (N,) true class indices
        lam: regularization penalty for low-ranked classes
        k_reg: rank threshold where penalty kicks in (1-indexed)
    
    Returns:
        scores: (N,) RAPS nonconformity scores
    """
    N, C = probs.shape
    sorted_probs, sorted_indices = torch.sort(probs, dim=1, descending=True)
    cumsum_probs = torch.cumsum(sorted_probs, dim=1)  # (N, C)
    
    # Find rank of true class (0-indexed position in sorted order)
    labels_expanded = labels.unsqueeze(1).expand_as(sorted_indices)
    true_class_mask = (sorted_indices == labels_expanded)  # (N, C) boolean
    
    # Get position (rank) of true class: shape (N,)
    ranks = true_class_mask.float().argmax(dim=1)  # 0-indexed
    
    # Cumulative probability at true class position
    cum_prob_at_true = cumsum_probs[true_class_mask]  # (N,)
    
    # RAPS penalty: λ * max(0, rank + 1 - k_reg) 
    penalty = lam * torch.clamp(ranks.float() + 1 - k_reg, min=0)
    
    scores = cum_prob_at_true + penalty
    return scores


def conformal_calibrate(val_probs, val_labels, alpha=0.05, lam=0.05, k_reg=4):
    """Calibrate RAPS conformal threshold on validation set.
    
    Args:
        val_probs: (N_val, C) softmax probabilities (temperature-scaled)
        val_labels: (N_val,) true labels
        alpha: miscoverage rate
        lam: RAPS lambda
        k_reg: RAPS k_reg
    
    Returns:
        q_hat: conformal threshold
    """
    scores = compute_raps_scores(val_probs, val_labels, lam=lam, k_reg=k_reg)
    n = len(scores)
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    q_hat = torch.quantile(scores, q_level).item()
    return q_hat


def conformal_predict(test_probs, q_hat, lam=0.05, k_reg=4):
    """Generate RAPS conformal prediction sets.
    
    Include classes in descending probability order until
    cumulative_prob + λ * max(0, rank - k_reg) exceeds q_hat.
    
    Args:
        test_probs: (N_test, C) softmax probabilities (temperature-scaled)
        q_hat: conformal threshold
        lam: RAPS lambda
        k_reg: RAPS k_reg
    
    Returns:
        prediction_sets: list of lists of class indices
        set_sizes: (N_test,) numpy array
    """
    N, C = test_probs.shape
    sorted_probs, sorted_indices = torch.sort(test_probs, dim=1, descending=True)
    cumsum_probs = torch.cumsum(sorted_probs, dim=1)
    
    prediction_sets = []
    set_sizes = []
    
    for i in range(N):
        n_include = C  # default: include all
        for j in range(C):
            rank = j + 1  # 1-indexed
            penalty = lam * max(0, rank - k_reg)
            raps_score = cumsum_probs[i, j].item() + penalty
            if raps_score >= q_hat:
                n_include = j + 1
                break
        
        pred_set = sorted_indices[i, :n_include].tolist()
        prediction_sets.append(pred_set)
        set_sizes.append(n_include)
    
    return prediction_sets, np.array(set_sizes)


def temperature_scale_probs(logits, temperature):
    """Apply temperature scaling to logits, then softmax."""
    return F.softmax(logits / temperature, dim=1)


def evaluate_conformal(prediction_sets, set_sizes, test_labels, alpha):
    """Evaluate conformal prediction quality."""
    test_labels_np = test_labels.numpy() if isinstance(test_labels, torch.Tensor) else test_labels
    covered = sum(1 for i, ps in enumerate(prediction_sets) if test_labels_np[i] in ps)
    empirical_coverage = covered / len(test_labels_np)
    singleton_rate = (set_sizes == 1).mean()
    
    return {
        "alpha": alpha,
        "target_coverage": 1 - alpha,
        "empirical_coverage": empirical_coverage,
        "avg_set_size": set_sizes.mean(),
        "median_set_size": np.median(set_sizes),
        "singleton_rate": singleton_rate,
        "set_sizes": set_sizes,
    }


print("RAPS conformal prediction functions defined.")
print(f"  Temperature scaling: applied BEFORE softmax")
print(f"  RAPS penalty: λ * max(0, rank - k_reg)")

In [ ]:
# Cell 5
# Run RAPS Conformal Prediction (temperature-scaled) 

conformal_results = {}
lam = CFG["raps_lambda"]
k_reg = CFG["raps_k_reg"]

for backbone_name in ["convnext_tiny", "deit_small"]:
    conformal_results[backbone_name] = {}
    data = saved_logits[backbone_name]
    
    for seed in CFG["seeds"]:
        conformal_results[backbone_name][seed] = {}
        
        val_logits = data["val_logits"][seed]
        val_labels = data["val_labels"]
        test_logits = data["test_logits"][seed]
        test_labels = data["test_labels"]
        
        # Apply temperature scaling
        T = temperatures[backbone_name][seed]
        val_probs = temperature_scale_probs(val_logits, T)
        test_probs = temperature_scale_probs(test_logits, T)
        
        for alpha in CFG["alpha_levels"]:
            q_hat = conformal_calibrate(val_probs, val_labels, alpha=alpha,
                                        lam=lam, k_reg=k_reg)
            pred_sets, set_sizes = conformal_predict(test_probs, q_hat,
                                                      lam=lam, k_reg=k_reg)
            result = evaluate_conformal(pred_sets, set_sizes, test_labels, alpha)
            result["q_hat"] = q_hat
            result["prediction_sets"] = pred_sets
            conformal_results[backbone_name][seed][alpha] = result
        
        print(f"\n{backbone_name} | seed {seed} (T={T:.4f}):")
        for alpha in CFG["alpha_levels"]:
            r = conformal_results[backbone_name][seed][alpha]
            print(f"  α={alpha:.2f} | Target={r['target_coverage']:.0%} | "
                  f"Emp.cov={r['empirical_coverage']:.1%} | "
                  f"Avg size={r['avg_set_size']:.2f} | "
                  f"Singletons={r['singleton_rate']:.1%}")

print("\nRAPS conformal prediction complete (temperature-scaled).")

In [ ]:
# Cell 6
# Conformal Prediction Summary Table 

print("=" * 80)
print("  CONFORMAL PREDICTION RESULTS")
print("=" * 80)

for backbone_name in ["convnext_tiny", "deit_small"]:
    print(f"\n--- {backbone_name} ---")
    print(f"{'α':>5} | {'Target Cov':>10} | {'Emp. Cov':>15} | {'Avg Set Size':>15} | {'Singletons':>15}")
    print("-" * 75)
    
    for alpha in CFG["alpha_levels"]:
        covs = [conformal_results[backbone_name][s][alpha]["empirical_coverage"]
                for s in CFG["seeds"]]
        sizes = [conformal_results[backbone_name][s][alpha]["avg_set_size"]
                 for s in CFG["seeds"]]
        sings = [conformal_results[backbone_name][s][alpha]["singleton_rate"]
                 for s in CFG["seeds"]]
        
        print(f"{alpha:>5.2f} | {1-alpha:>9.0%} | "
              f"{np.mean(covs):.1%} ± {np.std(covs):.1%} | "
              f"{np.mean(sizes):.2f} ± {np.std(sizes):.2f} | "
              f"{np.mean(sings):.1%} ± {np.std(sings):.1%}")

In [ ]:
# Cell 7
# AquaSelect-Conformal Fusion 

def aquaselect_conformal_fusion(
    aq_confidence,       # (N,) AquaSelect confidence from Notebook 2
    conformal_set_sizes, # (N,) conformal prediction set sizes
    test_probs,          # (N, C) softmax probabilities
    val_aq_confidence,   # (N_val,) AquaSelect confidence on val set
    val_set_sizes,       # (N_val,) conformal set sizes on val set
    val_probs,           # (N_val, C) softmax probabilities on val set
    val_labels,          # (N_val,) true labels
):
    """Fuse AquaSelect confidence with conformal set size.
    
    Features:
    1. AquaSelect confidence (selection head + UDAS fusion)
    2. 1 / conformal_set_size (smaller set = more confident)
    3. Max softmax probability
    
    Fit logistic regression on val set, apply to test set.
    """
    # Feature: inverse set size (normalized)
    max_set_size = max(val_set_sizes.max(), conformal_set_sizes.max())
    val_inv_size = 1.0 - (val_set_sizes - 1) / max(max_set_size - 1, 1)
    test_inv_size = 1.0 - (conformal_set_sizes - 1) / max(max_set_size - 1, 1)
    
    # Feature: max softmax prob
    val_max_prob = val_probs.max(dim=1).values.numpy()
    test_max_prob = test_probs.max(dim=1).values.numpy()
    
    # Build feature matrices
    val_features = np.stack([
        val_aq_confidence,
        val_inv_size,
        val_max_prob,
    ], axis=1)
    
    test_features = np.stack([
        aq_confidence,
        test_inv_size,
        test_max_prob,
    ], axis=1)
    
    # Target: was the prediction correct?
    val_preds = val_probs.argmax(dim=1).numpy()
    val_correct = (val_preds == val_labels.numpy()).astype(int)
    
    # Fit logistic regression
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(val_features, val_correct)
    
    test_confidence = lr.predict_proba(test_features)[:, 1]
    val_confidence = lr.predict_proba(val_features)[:, 1]
    
    print(f"    Fusion weights: {lr.coef_[0]}")
    print(f"    Features: [aq_confidence, inv_set_size, max_prob]")
    print(f"    Val accuracy: {lr.score(val_features, val_correct)*100:.1f}%")
    
    return test_confidence, val_confidence


print("AquaSelect-Conformal fusion function defined.")

In [ ]:
# Cell 8
# Run AquaSelect-Conformal Fusion 

def compute_risk_coverage_curve(predictions, targets, confidence_scores):
    sorted_indices = np.argsort(-confidence_scores)
    sorted_correct = (predictions == targets)[sorted_indices]
    n = len(sorted_correct)
    coverages, risks = [], []
    for k in range(1, n + 1):
        coverages.append(k / n)
        risks.append(1.0 - sorted_correct[:k].mean())
    aurcc = np.trapz(risks, coverages)
    return np.array(coverages), np.array(risks), aurcc


def coverage_at_accuracy(predictions, targets, confidence_scores, target_acc=0.95):
    sorted_indices = np.argsort(-confidence_scores)
    sorted_correct = (predictions == targets)[sorted_indices]
    n = len(sorted_correct)
    best_coverage = 0.0
    for k in range(1, n + 1):
        if sorted_correct[:k].mean() >= target_acc:
            best_coverage = k / n
    return best_coverage * 100


FUSION_ALPHA = 0.10  # 90% conformal coverage for fusion

aq_conformal_results = {}

print("=" * 70)
print("  AQUASELECT-CONFORMAL FUSION")
print("=" * 70)

for backbone_name in ["convnext_tiny", "deit_small"]:
    aq_conformal_results[backbone_name] = {}
    data = saved_logits[backbone_name]
    
    for seed in CFG["seeds"]:
        print(f"\n--- {backbone_name} | seed {seed} ---")
        
        val_logits = data["val_logits"][seed]
        val_labels = data["val_labels"]
        test_logits = data["test_logits"][seed]
        test_labels = data["test_labels"]
        
        val_probs = F.softmax(val_logits, dim=1)
        test_probs = F.softmax(test_logits, dim=1)
        
        # Temperature-scaled probs for conformal sets
        T = temperatures[backbone_name][seed]
        val_probs_ts = temperature_scale_probs(val_logits, T)
        test_probs_ts = temperature_scale_probs(test_logits, T)
        
        # Get conformal set sizes at fusion alpha (using temp-scaled + RAPS)
        lam = CFG["raps_lambda"]
        k_reg = CFG["raps_k_reg"]
        q_hat = conformal_calibrate(val_probs_ts, val_labels, alpha=FUSION_ALPHA,
                                    lam=lam, k_reg=k_reg)
        _, val_set_sizes = conformal_predict(val_probs_ts, q_hat, lam=lam, k_reg=k_reg)
        _, test_set_sizes = conformal_predict(test_probs_ts, q_hat, lam=lam, k_reg=k_reg)
        
        # Get AquaSelect confidence from Notebook 2
        aq_test_conf = nb2_results["aquaselect_results"][backbone_name][seed]["test_confidence"]
        
        # For val, use selection scores
        aq_val_conf = nb2_results["aquaselect_results"][backbone_name][seed].get(
            "val_sel_scores",
            F.softmax(val_logits, dim=1).max(dim=1).values.numpy()
        )
        
        # Run fusion
        test_confidence, val_confidence = aquaselect_conformal_fusion(
            aq_test_conf, test_set_sizes, test_probs,
            aq_val_conf, val_set_sizes, val_probs, val_labels,
        )
        
        # Evaluate
        predictions = test_logits.argmax(dim=1).numpy()
        targets = test_labels.numpy()
        
        coverages, risks, aurcc = compute_risk_coverage_curve(predictions, targets, test_confidence)
        cov95 = coverage_at_accuracy(predictions, targets, test_confidence, 0.95)
        cov99 = coverage_at_accuracy(predictions, targets, test_confidence, 0.99)
        
        aq_conformal_results[backbone_name][seed] = {
            "aurcc": aurcc,
            "cov95": cov95,
            "cov99": cov99,
            "coverages": coverages,
            "risks": risks,
            "test_confidence": test_confidence,
            "full_acc": (predictions == targets).mean() * 100,
        }
        
        print(f"  AURCC={aurcc:.4f} | Cov@95={cov95:.1f}% | Cov@99={cov99:.1f}%")
        
        # Compare to AquaSelect (no conformal)
        aq_aurcc = nb2_results["aquaselect_results"][backbone_name][seed]["aurcc"]
        delta = aurcc - aq_aurcc
        print(f"  vs AquaSelect: AURCC delta = {delta:+.4f} ({'better' if delta < 0 else 'worse'})")

print(f"\n{'=' * 70}")
print("AquaSelect-Conformal fusion complete.")

In [ ]:
# Cell 9
# Full Comparison Table 

sr_results = nb2_results["sr_results"]
mcd_results = nb2_results["mcd_results"]
de_results = nb2_results["de_results"]
aq_results = nb2_results["aquaselect_results"]


def method_summary(results_dict, seeds, is_single=False):
    if is_single:
        return (results_dict["aurcc"], 0,
                results_dict["cov95"], 0,
                results_dict["cov99"], 0)
    aurccs = [results_dict[s]["aurcc"] for s in seeds]
    cov95s = [results_dict[s]["cov95"] for s in seeds]
    cov99s = [results_dict[s]["cov99"] for s in seeds]
    return (np.mean(aurccs), np.std(aurccs),
            np.mean(cov95s), np.std(cov95s),
            np.mean(cov99s), np.std(cov99s))


for backbone_name in ["convnext_tiny", "deit_small"]:
    print(f"\n{'=' * 85}")
    print(f"  FULL COMPARISON -- {backbone_name}")
    print(f"{'=' * 85}")
    
    methods = [
        ("SR", sr_results[backbone_name], False),
        ("MCD", mcd_results[backbone_name], False),
        ("AquaSelect", aq_results[backbone_name], False),
        ("AQ-Conformal", aq_conformal_results[backbone_name], False),
        ("DE (3 models)", de_results[backbone_name], True),
    ]
    
    print(f"{'Method':>25} | {'AURCC ↓':>20} | {'Cov@95 (%) ↑':>18} | {'Cov@99 (%) ↑':>18} | {'Guarantee?':>10}")
    print("-" * 100)
    
    for name, rd, is_single in methods:
        a_m, a_s, c95_m, c95_s, c99_m, c99_s = method_summary(rd, CFG["seeds"], is_single)
        guarantee = "Yes (1-α)" if "Conformal" in name else "No"
        if is_single:
            print(f"{name:>25} | {a_m:>8.4f}             | {c95_m:>6.1f}              | {c99_m:>6.1f}              | {guarantee:>10}")
        else:
            print(f"{name:>25} | {a_m:.4f} ± {a_s:.4f}    | {c95_m:.1f} ± {c95_s:.1f}       | {c99_m:.1f} ± {c99_s:.1f}       | {guarantee:>10}")

In [ ]:
# Cell 10
# Selective Accuracy at Coverage Levels 

def selective_accuracy_at_coverages(predictions, targets, confidence, coverages):
    sorted_indices = np.argsort(-confidence)
    sorted_correct = (predictions == targets)[sorted_indices]
    n = len(sorted_correct)
    results = {}
    for cov in coverages:
        k = max(1, int(cov * n))
        results[cov] = sorted_correct[:k].mean() * 100
    return results


coverage_levels = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 1.00]
seed = 42

for backbone_name in ["convnext_tiny", "deit_small"]:
    print(f"\n{'=' * 80}")
    print(f"  SELECTIVE ACCURACY AT COVERAGE -- {backbone_name} (seed {seed})")
    print(f"{'=' * 80}")
    
    data = saved_logits[backbone_name]
    test_logits = data["test_logits"][seed]
    test_labels = data["test_labels"]
    predictions = test_logits.argmax(dim=1).numpy()
    targets = test_labels.numpy()
    
    # SR confidence
    sr_conf = F.softmax(test_logits, dim=1).max(dim=1).values.numpy()
    
    # AquaSelect confidence
    aq_conf = nb2_results["aquaselect_results"][backbone_name][seed]["test_confidence"]
    
    # AQ-Conformal confidence
    aqc_conf = aq_conformal_results[backbone_name][seed]["test_confidence"]
    
    # DE
    de_probs_list = [F.softmax(data["test_logits"][s], dim=1) for s in CFG["seeds"]]
    de_avg_probs = torch.stack(de_probs_list).mean(dim=0)
    de_preds = de_avg_probs.argmax(dim=1).numpy()
    de_entropy = -(de_avg_probs * torch.log(de_avg_probs + 1e-8)).sum(dim=1)
    de_conf = -de_entropy.numpy()
    
    # Print table
    header = f"{'Method':>15} |"
    for c in coverage_levels:
        header += f" {c*100:>5.0f}% |"
    print(header)
    print("-" * len(header))
    
    for method_name, preds, conf in [
        ("SR", predictions, sr_conf),
        ("AquaSelect", predictions, aq_conf),
        ("AQ-Conformal", predictions, aqc_conf),
        ("DE", de_preds, de_conf),
    ]:
        accs = selective_accuracy_at_coverages(preds, targets, conf, coverage_levels)
        row = f"{method_name:>15} |"
        for c in coverage_levels:
            row += f" {accs[c]:>5.1f}% |"
        print(row)

In [ ]:
# Cell 11
# Conformal Set Size Distribution 

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, backbone_name in zip(axes, ["convnext_tiny", "deit_small"]):
    r = conformal_results[backbone_name][42][0.10]
    sizes = r["set_sizes"]
    
    unique, counts = np.unique(sizes, return_counts=True)
    ax.bar(unique, counts / counts.sum() * 100, color="steelblue", edgecolor="white")
    ax.set_xlabel("Prediction Set Size", fontsize=12)
    ax.set_ylabel("Percentage (%)", fontsize=12)
    ax.set_title(f"{backbone_name.replace('_', ' ').title()}\n"
                 f"α=0.10 | Avg size={sizes.mean():.2f} | "
                 f"Singletons={100*(sizes==1).mean():.1f}%",
                 fontsize=11, fontweight="bold")
    ax.set_xticks(range(1, max(unique) + 1))

plt.suptitle("Conformal Prediction Set Size Distribution (Seed 42)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "conformal_set_size_distribution.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: conformal_set_size_distribution.png")

In [ ]:
# Cell 12
# Risk-Coverage Curves (All Methods Including AQ-Conformal) 

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

method_styles = {
    "SR":            {"color": "#888888", "ls": "--", "lw": 1.5},
    "MCD":           {"color": "#4477AA", "ls": "--", "lw": 1.5},
    "AquaSelect":    {"color": "#EE7733", "ls": "-",  "lw": 2.0},
    "AQ-Conformal":  {"color": "#CC3311", "ls": "-",  "lw": 2.5},
    "DE":            {"color": "#228833", "ls": "-.",  "lw": 1.5},
}

for ax, backbone_name in zip(axes, ["convnext_tiny", "deit_small"]):
    # Per-seed methods: plot mean curve
    for method_name, results_key in [
        ("SR", sr_results[backbone_name]),
        ("MCD", mcd_results[backbone_name]),
        ("AquaSelect", aq_results[backbone_name]),
        ("AQ-Conformal", aq_conformal_results[backbone_name]),
    ]:
        all_risks = [results_key[s]["risks"] for s in CFG["seeds"]]
        mean_risks = np.mean(all_risks, axis=0)
        std_risks = np.std(all_risks, axis=0)
        coverages = results_key[CFG["seeds"][0]]["coverages"]
        
        style = method_styles[method_name]
        ax.plot(coverages, mean_risks, color=style["color"], linestyle=style["ls"],
                linewidth=style["lw"], label=method_name)
        ax.fill_between(coverages, mean_risks - std_risks, mean_risks + std_risks,
                        color=style["color"], alpha=0.08)
    
    # DE: single curve
    de_r = de_results[backbone_name]
    style = method_styles["DE"]
    ax.plot(de_r["coverages"], de_r["risks"], color=style["color"],
            linestyle=style["ls"], linewidth=style["lw"], label="DE (3 models)")
    
    ax.set_xlabel("Coverage (φ)", fontsize=12)
    ax.set_ylabel("Selective Risk (Error Rate)", fontsize=12)
    ax.set_title(backbone_name.replace("_", " ").title(), fontsize=13, fontweight="bold")
    ax.legend(fontsize=9, loc="upper left")
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0, 0.25])
    ax.grid(True, alpha=0.2)
    
    ax.axhline(y=0.05, color="gray", linestyle=":", alpha=0.4, linewidth=0.8)
    ax.axhline(y=0.01, color="gray", linestyle=":", alpha=0.4, linewidth=0.8)

plt.suptitle("Risk-Coverage Curves (All Methods)", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "risk_coverage_with_conformal.png"),
            dpi=300, bbox_inches="tight")
plt.savefig(os.path.join(CFG["output_dir"], "risk_coverage_with_conformal.pdf"),
            bbox_inches="tight")
plt.show()
print("Saved: risk_coverage_with_conformal.png and .pdf")

In [ ]:
# Cell 13
# Per-Class Conformal Set Size (which species are most confusing?) 

backbone_name = "convnext_tiny"
seed = 42
alpha = 0.10

data = saved_logits[backbone_name]
test_labels = data["test_labels"].numpy()
r = conformal_results[backbone_name][seed][alpha]
set_sizes = r["set_sizes"]

# Mean set size per class
class_mean_sizes = []
class_counts = []
for c in range(CFG["num_classes"]):
    mask = test_labels == c
    if mask.sum() > 0:
        class_mean_sizes.append(set_sizes[mask].mean())
        class_counts.append(mask.sum())
    else:
        class_mean_sizes.append(0)
        class_counts.append(0)

# Sort by set size (hardest first)
sorted_classes = np.argsort(class_mean_sizes)[::-1]

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, CFG["num_classes"]))
bars = ax.barh(range(CFG["num_classes"]),
               [class_mean_sizes[c] for c in sorted_classes],
               color=[colors[i] for i in range(CFG["num_classes"])],
               edgecolor="white")

ax.set_yticks(range(CFG["num_classes"]))
ax.set_yticklabels([f"{label_names[c]} (n={class_counts[c]})" for c in sorted_classes], fontsize=9)
ax.set_xlabel("Mean Conformal Set Size", fontsize=12)
ax.set_title(f"Per-Class Conformal Set Size ({backbone_name}, α={alpha})\n"
             f"Larger set = more uncertain species", fontsize=12, fontweight="bold")
ax.axvline(x=1.0, color="green", linestyle=":", alpha=0.7, label="Perfect (size=1)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "conformal_perclass_setsize.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: conformal_perclass_setsize.png")

In [ ]:
# Cell 14
# Coverage Guarantee Verification 
# Empirical coverage should be >= target coverage for all configurations.

print("=" * 70)
print("  COVERAGE GUARANTEE VERIFICATION")
print("  (Empirical coverage should be ≥ target for all configs)")
print("=" * 70)

all_pass = True
rows = []

for backbone_name in ["convnext_tiny", "deit_small"]:
    for seed in CFG["seeds"]:
        for alpha in CFG["alpha_levels"]:
            r = conformal_results[backbone_name][seed][alpha]
            target = 1 - alpha
            emp = r["empirical_coverage"]
            passed = emp >= target - 0.001  # small float tolerance
            if not passed:
                all_pass = False
            rows.append({
                "Backbone": backbone_name,
                "Seed": seed,
                "α": alpha,
                "Target": f"{target:.0%}",
                "Empirical": f"{emp:.1%}",
                "Pass": "✓" if passed else "✗",
            })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nAll guarantees met: {'YES ✓' if all_pass else 'NO ✗'}")

In [ ]:
# Cell 15
# Save all results 

output_dir = CFG["output_dir"]

save_data = {
    "conformal_results": {},
    "aq_conformal_results": {},
    "config": CFG,
    "fusion_alpha": FUSION_ALPHA,
}

for backbone_name in ["convnext_tiny", "deit_small"]:
    save_data["conformal_results"][backbone_name] = {}
    for seed in CFG["seeds"]:
        save_data["conformal_results"][backbone_name][seed] = {}
        for alpha in CFG["alpha_levels"]:
            r = conformal_results[backbone_name][seed][alpha]
            save_data["conformal_results"][backbone_name][seed][alpha] = {
                "empirical_coverage": r["empirical_coverage"],
                "avg_set_size": r["avg_set_size"],
                "singleton_rate": r["singleton_rate"],
                "set_sizes": r["set_sizes"],
                "q_hat": r["q_hat"],
            }
    
    save_data["aq_conformal_results"][backbone_name] = {}
    for seed in CFG["seeds"]:
        r = aq_conformal_results[backbone_name][seed]
        save_data["aq_conformal_results"][backbone_name][seed] = {
            "aurcc": r["aurcc"],
            "cov95": r["cov95"],
            "cov99": r["cov99"],
            "coverages": r["coverages"],
            "risks": r["risks"],
            "test_confidence": r["test_confidence"],
        }

results_path = os.path.join(output_dir, "notebook4_conformal_results.pth")
torch.save(save_data, results_path)
print(f"Saved: {results_path} ({os.path.getsize(results_path) / 1e6:.1f} MB)")

# List outputs
print(f"\nAll outputs in {output_dir}:")
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    if os.path.isfile(fpath):
        print(f"  {f} ({os.path.getsize(fpath) / 1e6:.1f} MB)")

In [ ]:
# Cell 16
print("Notebook 4 complete.")